## Importing libraries

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import plotly.express as px
print('Done')

Done


### Generating dataset for sales dashboard

In [36]:
np.random.seed(42)
random.seed(42)
categories = {
    'Electronics':  ['Laptop Pro 15', 'Wireless Mouse', 'USB Hub', 'Monitor 27"'],
    'Clothing':     ['Polo Shirt', 'Denim Jeans', 'Running Shoes', 'Cap'],
    'Food & Bev':   ['Coffee Blend 500g', 'Protein Bar Box', 'Energy Drink'],
    'Home & Living': ['Desk Lamp', 'Storage Box', 'Wall Clock'],
    'Sports':       ['Yoga Mat', 'Resistance Bands', 'Jump Rope']
}
prices = {
    'Laptop Pro 15': 65000, 'Wireless Mouse': 850, 'USB Hub': 1200, 'Monitor 27"': 18000,
    'Polo Shirt': 650,  'Denim Jeans': 1800, 'Running Shoes': 3500, 'Cap': 400,
    'Coffee Blend 500g': 380, 'Protein Bar Box': 950, 'Energy Drink': 75,
    'Desk Lamp': 1200, 'Storage Box': 550, 'Wall Clock': 800,
    'Yoga Mat': 1500, 'Resistance Bands': 600, 'Jump Rope': 350
}
regions    = ['NCR', 'Luzon', 'Visayas', 'Mindanao']
payments   = ['Credit Card', 'GCash', 'Cash on Delivery', 'Bank Transfer']
customers  = [f'Customer_{i:03d}' for i in range(1, 81)]
 


In [37]:
# ── Build rows ────────────────────────────────────────────────
rows = []
start = datetime(2024, 1, 1)
for i in range(1, 1001):                       # 1 000 transactions
    cat     = random.choice(list(categories))
    product = random.choice(categories[cat])
    qty     = random.randint(1, 5)
    price   = prices[product]
    date    = start + timedelta(days=random.randint(0, 364))
    rows.append({
        'order_id':       f'ORD-{1000+i}',
        'order_date':     date.strftime('%Y-%m-%d'),
        'customer_name':  random.choice(customers),
        'region':         random.choice(regions),
        'category':       cat,
        'product_name':   product,
        'quantity':       qty,
        'unit_price':     price,
        'total_amount':   qty * price,
        'payment_method': random.choice(payments)
    })
 
df = pd.DataFrame(rows)
#

In [4]:
df.head()

,order_id,order_date,customer_name,region,category,product_name,quantity,unit_price,total_amount,payment_method,year,month,month_name,month_order
0,ORD-1001,2024-05-05,Customer_029,Luzon,Electronics,Laptop Pro 15,3,65000.0,195000.0,Credit Card,2024,5,May 2024,2024-05
1,ORD-1002,2024-08-04,Customer_005,NCR,Sports,Yoga Mat,5,1500.0,7500.0,Credit Card,2024,8,Aug 2024,2024-08
2,ORD-1003,2024-11-04,Customer_004,Luzon,Clothing,Denim Jeans,5,1800.0,9000.0,Bank Transfer,2024,11,Nov 2024,2024-11
3,ORD-1004,2024-05-22,Customer_001,Luzon,Clothing,Cap,5,400.0,2000.0,Bank Transfer,2024,5,May 2024,2024-05
4,ORD-1005,2024-04-20,Customer_044,NCR,Food & Bev,Protein Bar Box,2,950.0,1900.0,Credit Card,2024,4,Apr 2024,2024-04


### Saving dataframe into csv

In [ ]:
df.to_csv('asset/data/sales_data.csv', index=False)


In [40]:
print(f'Generated {len(df)} rows -> asset/data/sales_data.csv')

Generated 1000 rows -> asset/data/sales_data.csv


### Data Preparation

In [3]:
# ── 1. Load CSV ───────────────────────────────────────────────
df = pd.read_csv('asset/data/sales_data.csv')
 
# ── 2. Parse dates ────────────────────────────────────────────
# Convert the order_date column from string to proper datetime
df['order_date'] = pd.to_datetime(df['order_date'])
 
# ── 3. Extract helper columns ─────────────────────────────────
df['year']        = df['order_date'].dt.year
df['month']       = df['order_date'].dt.month
df['month_name']  = df['order_date'].dt.strftime('%b %Y')  # e.g. Jan 2024
df['month_order'] = df['order_date'].dt.to_period('M')     # for correct sorting
 
# ── 4. Handle missing values ──────────────────────────────────
df['total_amount'] = df['total_amount'].fillna(df['quantity'] * df['unit_price'])
df.dropna(subset=['order_id', 'order_date'], inplace=True)   # drop critical nulls
 
# ── 5. Ensure correct data types ──────────────────────────────
df['quantity']     = df['quantity'].astype(int)
df['unit_price']   = df['unit_price'].astype(float)
df['total_amount'] = df['total_amount'].astype(float)
 
# ── 6. Quick sanity check (printed to terminal on startup) ────
print(f'Loaded {len(df)} rows, date range: {df.order_date.min().date()} to {df.order_date.max().date()}')

Loaded 1000 rows, date range: 2024-01-01 to 2024-12-30


In [5]:
# ── Monthly revenue trend ─────────────────────────────────────
monthly = (
    df.groupby(['month_order', 'month_name'], as_index=False)
      .agg(revenue=('total_amount', 'sum'), orders=('order_id', 'count'))
      .sort_values('month_order')
)
 
# ── Revenue by category ───────────────────────────────────────
by_category = (
    df.groupby('category', as_index=False)
      .agg(revenue=('total_amount', 'sum'), orders=('order_id', 'count'))
      .sort_values('revenue', ascending=False)
)
 
# ── Revenue by region ─────────────────────────────────────────
by_region = (
    df.groupby('region', as_index=False)
      .agg(revenue=('total_amount', 'sum'))
)
 
# ── Top 10 customers ──────────────────────────────────────────
top_customers = (
    df.groupby('customer_name', as_index=False)
      .agg(revenue=('total_amount', 'sum'), orders=('order_id', 'count'))
      .sort_values('revenue', ascending=False)
      .head(10)
)


### Displaying of graphs

In [6]:
import plotly.graph_objects as go
import plotly.express as px
 
def make_revenue_trend(dataframe):
    """Line chart: monthly total revenue with markers."""
    monthly = (
        dataframe.groupby(['month_order', 'month_name'], as_index=False)
                 .agg(revenue=('total_amount', 'sum'))
                 .sort_values('month_order')
    )
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=monthly['month_name'],
        y=monthly['revenue'],
        mode='lines+markers',         # line with dots at each data point
        name='Monthly Revenue',
        line=dict(color='#2E75B6', width=3),
        marker=dict(size=8, color='#1F4E79'),
        fill='tozeroy',               # shade area under the line
        fillcolor='rgba(46,117,182,0.1)'
    ))
    fig.update_layout(
        title='Monthly Revenue Trend',
        xaxis_title='Month',
        yaxis_title='Revenue (PHP)',
        plot_bgcolor='white',
        paper_bgcolor='white',
        hovermode='x unified',        # show all values at hover x position
        yaxis=dict(tickformat=',.0f') # format numbers with commas
    )
    return fig


In [9]:
fig_revenue_trend = make_revenue_trend(df)
fig_revenue_trend.show()

In [10]:
def make_category_bar(dataframe):
    """Horizontal bar chart: revenue per product category."""
    by_cat = (
        dataframe.groupby('category', as_index=False)
                 .agg(revenue=('total_amount', 'sum'))
                 .sort_values('revenue')      # ascending so largest is on top
    )
    fig = px.bar(
        by_cat,
        x='revenue',
        y='category',
        orientation='h',              # horizontal bars
        color='revenue',
        color_continuous_scale='Blues',
        title='Revenue by Category',
        labels={'revenue': 'Revenue (PHP)', 'category': ''},
        text_auto=',.0f'              # show value labels on bars
    )
    fig.update_layout(coloraxis_showscale=False, plot_bgcolor='white')
    return fig



In [11]:
fig_category_bar = make_category_bar(df)
fig_category_bar.show()

In [ ]:
def make_region_pie(dataframe):
    """Donut pie chart: revenue share by region."""
    by_region = (
        dataframe.groupby('region', as_index=False)
                 .agg(revenue=('total_amount', 'sum'))
    )
    fig = px.pie(
        by_region,
        names='region',
        values='revenue',
        title='Revenue by Region',
        hole=0.4,                     # 0.4 creates a donut shape
        color_discrete_sequence=px.colors.sequential.Blues_r
    )
    fig.update_traces(textposition='outside', textinfo='percent+label')
    return fig


In [13]:
fig_region_pie = make_region_pie(df)
fig_region_pie.show()

In [14]:
def make_top_customers(dataframe):
    """Bar chart: top 10 customers by total spend."""
    top = (
        dataframe.groupby('customer_name', as_index=False)
                 .agg(revenue=('total_amount', 'sum'))
                 .sort_values('revenue', ascending=False)
                 .head(10)
                 .sort_values('revenue')        # flip for chart orientation
    )
    fig = px.bar(
        top,
        x='revenue', y='customer_name',
        orientation='h',
        title='Top 10 Customers by Revenue',
        labels={'revenue': 'Total Spend (PHP)', 'customer_name': ''},
        color='revenue',
        color_continuous_scale='Blues',
        text_auto=',.0f'
    )
    fig.update_layout(coloraxis_showscale=False, plot_bgcolor='white')
    return fig


In [15]:
fig_top_customers = make_top_customers(df)
fig_top_customers.show()

In [17]:
import dash
from dash import dcc, html, dash_table, Input, Output
import dash_bootstrap_components as dbc
 
# Initialize the app with a Bootstrap theme for better default styling
app = dash.Dash(
    __name__,
    external_stylesheets=[dbc.themes.BOOTSTRAP],
    title='Sales Dashboard'
)

## Creating Dashboard using Python Dash

In [18]:
def kpi_card(title, value, icon, color):
    """Return a Bootstrap card component with a KPI metric."""
    return dbc.Card([
        dbc.CardBody([
            html.Div([
                html.I(className=f'bi {icon} fs-2', style={'color': color}),
                html.Div([
                    html.P(title, className='text-muted mb-0', style={'fontSize': '0.85rem'}),
                    html.H4(value, className='mb-0 fw-bold', style={'color': color})
                ], className='ms-3')
            ], className='d-flex align-items-center')
        ])
    ], className='shadow-sm h-100')

In [19]:
app.layout = dbc.Container([
 
    # ── Header bar ─────────────────────────────────────────────
    dbc.Row([
        dbc.Col(
            html.H2('Sales Transaction Dashboard',
                    className='text-white fw-bold py-3 mb-0'),
            style={'background': '#1F4E79'}
        )
    ], className='mb-4'),
 
    # ── Filter row ─────────────────────────────────────────────
    dbc.Row([
        dbc.Col([
            html.Label('Filter by Region:', className='fw-semibold'),
            dcc.Dropdown(
                id='region-filter',
                options=[{'label': r, 'value': r} for r in sorted(df['region'].unique())],
                multi=True,
                placeholder='All Regions...'
            )
        ], md=4),
        dbc.Col([
            html.Label('Filter by Category:', className='fw-semibold'),
            dcc.Dropdown(
                id='category-filter',
                options=[{'label': c, 'value': c} for c in sorted(df['category'].unique())],
                multi=True,
                placeholder='All Categories...'
            )
        ], md=4),
        dbc.Col([
            html.Label('Date Range:', className='fw-semibold'),
            dcc.DatePickerRange(
                id='date-filter',
                start_date=df['order_date'].min(),
                end_date=df['order_date'].max(),
                display_format='MMM DD, YYYY'
            )
        ], md=4)
    ], className='mb-4 p-3 bg-light rounded'),
 
    # ── KPI cards row ──────────────────────────────────────────
    dbc.Row(id='kpi-row', className='mb-4'),
    # ── Charts row 1: trend + pie ──────────────────────────────
    dbc.Row([
        dbc.Col(dcc.Graph(id='revenue-trend'), md=8),
        dbc.Col(dcc.Graph(id='region-pie'),    md=4)
    ], className='mb-4'),
 
    # ── Charts row 2: category bar + top customers ─────────────
    dbc.Row([
        dbc.Col(dcc.Graph(id='category-bar'),   md=6),
        dbc.Col(dcc.Graph(id='top-customers'),  md=6)
    ], className='mb-4'),
 
    # ── Data table ─────────────────────────────────────────────
    dbc.Row([
        dbc.Col([
            html.H5('Transaction Details', className='fw-bold mb-3'),
            dash_table.DataTable(
                id='data-table',
                columns=[{'name': c.replace('_',' ').title(), 'id': c}
                         for c in ['order_id','order_date','customer_name',
                                   'region','category','product_name',
                                   'quantity','unit_price','total_amount']],
                page_size=15,
                sort_action='native',
                filter_action='native',
                style_header={'backgroundColor': '#1F4E79',
                              'color': 'white', 'fontWeight': 'bold'},
                style_data_conditional=[
                    {'if': {'row_index': 'odd'}, 'backgroundColor': '#F0F4F8'}
                ],
                style_cell={'textAlign': 'left', 'padding': '8px',
                            'fontFamily': 'Arial', 'fontSize': '13px'}
            )
        ])
    ])
 
], fluid=True)



In [20]:
@app.callback(
    Output('kpi-row',       'children'),
    Output('revenue-trend', 'figure'),
    Output('region-pie',    'figure'),
    Output('category-bar',  'figure'),
    Output('top-customers', 'figure'),
    Output('data-table',    'data'),
    Input('region-filter',   'value'),
    Input('category-filter', 'value'),
    Input('date-filter',     'start_date'),
    Input('date-filter',     'end_date'),
)
def update_dashboard(regions, categories, start_date, end_date):
    """
    Fires when any filter changes.
    Parameters arrive as Python objects (list, string, or None).
    Returns updated figures and table data.
    """
    # ── Apply filters ──────────────────────────────────────────
    filtered = df.copy()

    if regions:      # list of selected regions (None = all)
        filtered = filtered[filtered['region'].isin(regions)]

    if categories:   # list of selected categories
        filtered = filtered[filtered['category'].isin(categories)]

    if start_date:   # date string from picker e.g. '2024-03-01'
        filtered = filtered[filtered['order_date'] >= start_date]

    if end_date:
        filtered = filtered[filtered['order_date'] <= end_date]

    # ── Compute KPIs ───────────────────────────────────────────
    total_rev   = filtered['total_amount'].sum()
    total_orders = len(filtered)
    avg_order   = total_rev / total_orders if total_orders else 0
    total_units  = filtered['quantity'].sum()

    kpis = dbc.Row([
        dbc.Col(kpi_card('Total Revenue',
                         f'PHP {total_rev:,.0f}',
                         'bi-currency-dollar', '#1F4E79'), md=3),
        dbc.Col(kpi_card('Total Orders',
                         f'{total_orders:,}',
                         'bi-receipt', '#2E75B6'), md=3),
        dbc.Col(kpi_card('Avg Order Value',
                         f'PHP {avg_order:,.0f}',
                         'bi-graph-up', '#17A2B8'), md=3),
        dbc.Col(kpi_card('Units Sold',
                         f'{total_units:,}',
                         'bi-box-seam', '#28A745'), md=3),
    ])

    # ── Build charts from filtered data ───────────────────────
    trend    = make_revenue_trend(filtered)
    pie      = make_region_pie(filtered)
    bar      = make_category_bar(filtered)
    top_cust = make_top_customers(filtered)

    # ── Prepare table data ─────────────────────────────────────
    table_df = filtered[['order_id','order_date','customer_name',
                          'region','category','product_name',
                          'quantity','unit_price','total_amount']].copy()
    table_df['order_date']   = table_df['order_date'].dt.strftime('%Y-%m-%d')
    table_df['unit_price']   = table_df['unit_price'].map('{:,.2f}'.format)
    table_df['total_amount'] = table_df['total_amount'].map('{:,.2f}'.format)

    return kpis, trend, pie, bar, top_cust, table_df.to_dict('records')

In [24]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import dash
from dash import dcc, html, dash_table, Input, Output
import dash_bootstrap_components as dbc

# ── Load & clean data ──────────────────────────────────────────
df = pd.read_csv('asset/data/sales_data.csv')
df['order_date']  = pd.to_datetime(df['order_date'])
df['month_name']  = df['order_date'].dt.strftime('%b %Y')
df['month_order'] = df['order_date'].dt.to_period('M')
df['total_amount'] = df['total_amount'].fillna(df['quantity'] * df['unit_price'])

In [25]:
# ── Chart factory functions ────────────────────────────────────
def make_revenue_trend(d):
    m = (d.groupby(['month_order','month_name'], as_index=False)
          .agg(revenue=('total_amount','sum')).sort_values('month_order'))
    fig = go.Figure(go.Scatter(
        x=m['month_name'], y=m['revenue'],
        mode='lines+markers', fill='tozeroy',
        line=dict(color='#2E75B6', width=3),
        marker=dict(size=8), fillcolor='rgba(46,117,182,0.1)'
    ))
    fig.update_layout(title='Monthly Revenue Trend',
        xaxis_title='Month', yaxis_title='Revenue (PHP)',
        plot_bgcolor='white', hovermode='x unified',
        yaxis=dict(tickformat=',.0f'))
    return fig

def make_category_bar(d):
    b = (d.groupby('category', as_index=False)
          .agg(revenue=('total_amount','sum')).sort_values('revenue'))
    fig = px.bar(b, x='revenue', y='category', orientation='h',
        color='revenue', color_continuous_scale='Blues',
        title='Revenue by Category', text_auto=',.0f',
        labels={'revenue':'Revenue (PHP)','category':''})
    fig.update_layout(coloraxis_showscale=False, plot_bgcolor='white')
    return fig

def make_region_pie(d):
    r = d.groupby('region', as_index=False).agg(revenue=('total_amount','sum'))
    return px.pie(r, names='region', values='revenue', hole=0.4,
        title='Revenue by Region',
        color_discrete_sequence=px.colors.sequential.Blues_r)

def make_top_customers(d):
    t = (d.groupby('customer_name', as_index=False)
          .agg(revenue=('total_amount','sum'))
          .sort_values('revenue', ascending=False).head(10)
          .sort_values('revenue'))
    fig = px.bar(t, x='revenue', y='customer_name', orientation='h',
        color='revenue', color_continuous_scale='Blues',
        title='Top 10 Customers', text_auto=',.0f',
        labels={'revenue':'Total Spend (PHP)','customer_name':''})
    fig.update_layout(coloraxis_showscale=False, plot_bgcolor='white')
    return fig
    
def kpi_card(title, value, icon, color):
    return dbc.Card([dbc.CardBody([
        html.Div([
            html.I(className=f'bi {icon} fs-2', style={'color': color}),
            html.Div([
                html.P(title, className='text-muted mb-0',
                       style={'fontSize':'0.85rem'}),
                html.H4(value, className='mb-0 fw-bold', style={'color': color})
            ], className='ms-3')
        ], className='d-flex align-items-center')
    ])], className='shadow-sm h-100')

# ── App init ───────────────────────────────────────────────────
app = dash.Dash(__name__,
    external_stylesheets=[
        dbc.themes.BOOTSTRAP,
        'https://cdn.jsdelivr.net/npm/bootstrap-icons/font/bootstrap-icons.css'
    ],
    title='Sales Dashboard')

# ── Layout ─────────────────────────────────────────────────────
app.layout = dbc.Container([
    dbc.Row([dbc.Col(html.H2('Sales Transaction Dashboard',
        className='text-white fw-bold py-3 mb-0'),
        style={'background':'#1F4E79'})], className='mb-4'),
    dbc.Row([
        dbc.Col([html.Label('Region:'), dcc.Dropdown(
            id='region-filter', multi=True, placeholder='All...',
            options=[{'label':r,'value':r} for r in sorted(df.region.unique())])], md=4),
        dbc.Col([html.Label('Category:'), dcc.Dropdown(
            id='category-filter', multi=True, placeholder='All...',
            options=[{'label':c,'value':c} for c in sorted(df.category.unique())])], md=4),
        dbc.Col([html.Label('Dates:'), dcc.DatePickerRange(
            id='date-filter',
            start_date=df.order_date.min(), end_date=df.order_date.max(),
            display_format='MMM DD, YYYY')], md=4),
    ], className='mb-4 p-3 bg-light rounded'),
    dbc.Row(id='kpi-row', className='mb-4'),
    dbc.Row([dbc.Col(dcc.Graph(id='revenue-trend'), md=8),
             dbc.Col(dcc.Graph(id='region-pie'),    md=4)], className='mb-4'),
    dbc.Row([dbc.Col(dcc.Graph(id='category-bar'),  md=6),
             dbc.Col(dcc.Graph(id='top-customers'), md=6)], className='mb-4'),
    dbc.Row([dbc.Col([
        html.H5('Transaction Details', className='fw-bold mb-3'),
        dash_table.DataTable(
            id='data-table', page_size=15,
            sort_action='native', filter_action='native',
            columns=[{'name':c.replace('_',' ').title(),'id':c}
                     for c in ['order_id','order_date','customer_name',
                               'region','category','product_name',
                               'quantity','unit_price','total_amount']],
            style_header={'backgroundColor':'#1F4E79',
                          'color':'white','fontWeight':'bold'},
            style_data_conditional=[{'if':{'row_index':'odd'},
                                     'backgroundColor':'#F0F4F8'}],
            style_cell={'textAlign':'left','padding':'8px',
                        'fontFamily':'Arial','fontSize':'13px'}
        )
    ])])
], fluid=True)

# ── Callback ───────────────────────────────────────────────────
@app.callback(
    Output('kpi-row','children'),      Output('revenue-trend','figure'),
    Output('region-pie','figure'),      Output('category-bar','figure'),
    Output('top-customers','figure'),   Output('data-table','data'),
    Input('region-filter','value'),     Input('category-filter','value'),
    Input('date-filter','start_date'),  Input('date-filter','end_date'),
)
def update_dashboard(regions, categories, start_date, end_date):
    f = df.copy()
    if regions:    f = f[f.region.isin(regions)]
    if categories: f = f[f.category.isin(categories)]
    if start_date: f = f[f.order_date >= start_date]
    if end_date:   f = f[f.order_date <= end_date]

    rev    = f.total_amount.sum()
    orders = len(f)
    kpis = dbc.Row([
        dbc.Col(kpi_card('Total Revenue',   f'PHP {rev:,.0f}',
                         'bi-currency-dollar','#1F4E79'), md=3),
        dbc.Col(kpi_card('Total Orders',    f'{orders:,}',
                         'bi-receipt','#2E75B6'), md=3),
        dbc.Col(kpi_card('Avg Order Value', f'PHP {rev/orders:,.0f}' if orders else 'N/A',
                         'bi-graph-up','#17A2B8'), md=3),
        dbc.Col(kpi_card('Units Sold',      f'{f.quantity.sum():,}',
                         'bi-box-seam','#28A745'), md=3),
    ])
    td = f[['order_id','order_date','customer_name','region',
             'category','product_name','quantity','unit_price',
             'total_amount']].copy()
    td['order_date']   = td.order_date.dt.strftime('%Y-%m-%d')
    td['unit_price']   = td.unit_price.map('{:,.2f}'.format)
    td['total_amount'] = td.total_amount.map('{:,.2f}'.format)
    return kpis, make_revenue_trend(f), make_region_pie(f), \
           make_category_bar(f), make_top_customers(f), td.to_dict('records')

if __name__ == '__main__':
    app.run(debug=True)   # debug=True enables hot reload